# Custosell — Lightweight POS Database Schema
**Version:** 2.1  
**Date:** 2026-06-02  
**Purpose:** Lightweight POS system for shops, restaurants, salons, pharmacies in Uganda — daily sales tracking, inventory, shift management, expenses, role-based access, and subscription plans.
**Default Currency:** UGX (Ugandan Shillings)

---
## 📊 Table Overview

| # | Table | Purpose |
|---|-------|---------|
| 1 | `plans` | Subscription plans with JSON features & limits |
| 2 | `subscriptions` | Business subscription tracking |
| 3 | `businesses` | Business registration (replaces Custocare's facilities) |
| 4 | `users` | Staff/users with password auth, linked to a role |
| 5 | `roles` | Business-scoped roles with JSON permissions |
| 6 | `categories` | Product groupings |
| 7 | `products` | Inventory items for sale |
| 8 | `customers` | Buyer tracking |
| 9 | `sales` | Transaction header |
| 10 | `sale_items` | Line items per sale |
| 11 | `stock_movements` | Inventory ledger (audit trail) |
| 12 | `shifts` | User work session tracking |
| 13 | `expense_categories` | Categories for expenses |
| 14 | `expenses` | Business expense tracking |

---

## 1. `plans`
**Purpose:** Define subscription tiers. Seeded at setup — admins can add/modify.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | increments | PK | |
| name | string(100) | required | e.g. "Free", "Pro", "Premium" |
| slug | string(100) | unique, required | e.g. "free", "pro", "premium" |
| description | text | nullable | Marketing copy |
| price_monthly | decimal(10,2) | default 0 | UGX |
| price_yearly | decimal(10,2) | nullable | Discounted yearly rate |
| features | JSON | required | Feature flags |
| limits | JSON | required | Numerical limits |
| is_active | boolean | default true | |
| sort_order | integer | default 0 | Display order |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

### Seeded plans:

#### 🆓 Free — UGX 0/mo
```json
{
  "features": {
    "expenses": false,
    "shift_tracking": false,
    "discounts": false,
    "refunds": false,
    "export_data": false
  },
  "limits": {
    "staff_users": 1,
    "products": 50,
    "monthly_sales": 100,
    "customers": 50,
    "categories": 5
  }
}
```
**Reports:** Daily sales only

#### ⭐ Pro — UGX 30,000/mo
```json
{
  "features": {
    "expenses": true,
    "shift_tracking": true,
    "discounts": true,
    "refunds": true,
    "export_data": false
  },
  "limits": {
    "staff_users": 5,
    "products": 1000,
    "monthly_sales": null,
    "customers": 1000,
    "categories": 30
  }
}
```
**Reports:** Daily + monthly + net profit

#### 🏢 Premium — UGX 100,000/mo
```json
{
  "features": {
    "expenses": true,
    "shift_tracking": true,
    "discounts": true,
    "refunds": true,
    "export_data": true
  },
  "limits": {
    "staff_users": null,
    "products": null,
    "monthly_sales": null,
    "customers": null,
    "categories": null
  }
}
```
**Reports:** Full + CSV export

`null` limit = unlimited

## 2. `subscriptions`
**Purpose:** Track which plan each business is on and billing status.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | increments | PK | |
| business_id | bigInteger | FK → businesses.id, unique | One active sub per business |
| plan_id | bigInteger | FK → plans.id, required | Current plan |
| status | enum('active','trialing','cancelled','expired') | required | |
| starts_at | datetime | required | When subscription began |
| trial_ends_at | datetime | nullable | Free trial expiration |
| ends_at | datetime | nullable | When access stops |
| cancelled_at | datetime | nullable | When user cancelled |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** unique on `business_id`, index on `plan_id`, index on `status`

**Relationships:**
- `Subscription` **belongs to** `Business`
- `Subscription` **belongs to** `Plan`

## 3. `businesses`
**Purpose:** Each tenant/business that uses Custosell. Replaces the Facility model from Custocare.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| owner_id | bigInteger | FK → users.id, required | The user who owns this business |
| name | string(255) | required | Business name |
| slug | string(255) | unique, required | URL-friendly identifier |
| email | string(255) | nullable | Business contact email |
| phone | string(50) | nullable | Business contact phone |
| address | text | nullable | Physical address |
| currency | string(10) | default 'UGX' | Ugandan Shillings |
| receipt_footer | text | nullable | Custom message on receipts |
| logo_path | string(255) | nullable | Business logo |
| status | enum('active','suspended') | default 'active' | Account status |
| trial_ends_at | datetime | nullable | Quick access without join |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |
| deleted_at | timestamp | nullable | soft deletes |

**Indexes:** unique on `slug`, index on `owner_id`

**Relationships:**
- `Business` **belongs to** `User` (as owner — the person who owns it)
- `Business` **has one** `Subscription`
- `Business` **belongs to** `Plan` (via subscription)
- `Business` **has many** `Users` (staff)
- `Business` **has many** `Roles`
- `Business` **has many** `Categories`
- `Business` **has many** `Products`
- `Business` **has many** `Customers`
- `Business` **has many** `Sales`
- `Business` **has many** `StockMovements`
- `Business` **has many** `Shifts`
- `Business` **has many** `ExpenseCategories`
- `Business` **has many** `Expenses`

## 4. `users`
**Purpose:** Staff/users of a business. Password-based auth. Business owner can add/remove staff.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, nullable | Null for system-level (e.g. super admin) |
| role_id | bigInteger | FK → roles.id, nullable | Business-scoped role |
| name | string(255) | required | Full name |
| email | string(255) | unique, required | Login credential |
| password | string(255) | required | Hashed password |
| is_active | boolean | default true | Can this user log in? Owner can deactivate |
| phone | string(50) | nullable | |
| created_by | bigInteger | FK → users.id, nullable | Who added this user (owner/admin) |
| email_verified_at | timestamp | nullable | |
| remember_token | string(100) | nullable | |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |
| deleted_at | timestamp | nullable | soft deletes — owner can "remove" staff |

**Indexes:** unique on `email`, index on `business_id`, index on `role_id`, index on `created_by`

**Relationships:**
- `User` **belongs to** `Business` (nullable — the business they work for)
- `User` **has one** `Business` (as owned_business, if they're an owner)
- `User` **belongs to** `Role` (nullable)
- `User` **belongs to** `User` (as created_by, nullable — the admin who added them)
- `User` **has many** `Sales` (processed by this user)
- `User` **has many** `Shifts`
- `User` **has many** `StockMovements` (created_by)
- `User` **has many** `Expenses` (recorded_by)

## 5. `roles`
**Purpose:** Business-scoped roles with JSON-based permissions. Owner can manage who sees what.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, nullable | Null for system default roles |
| name | string(100) | required | e.g. "Admin", "Staff", "Cashier" |
| slug | string(100) | required | e.g. "admin", "staff" |
| description | text | nullable | Role description |
| permissions | JSON | required | JSON object of permission flags |
| is_default | boolean | default false | Auto-assigned to new users? |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** unique on `(business_id, slug)`

**Default permissions structure (JSON):**
```json
{
  "sales.create": true,
  "sales.view": true,
  "sales.refund": false,
  "sales.discount": false,
  "sales.delete": false,
  "inventory.view": true,
  "inventory.create": false,
  "inventory.edit": false,
  "inventory.delete": false,
  "customers.view": true,
  "customers.create": true,
  "customers.edit": false,
  "expenses.view": false,
  "expenses.create": false,
  "expenses.edit": false,
  "expenses.delete": false,
  "users.view": false,
  "users.create": false,
  "users.edit": false,
  "users.delete": false,
  "reports.view": false,
  "settings.view": false,
  "settings.edit": false
}
```

**Seeded roles per business:**
- **Admin** — all permissions `true` (for business owner)
- **Staff** — limited: can create sales, view inventory, manage customers, but cannot refund, cannot manage users, cannot see expenses/reports

**Relationships:**
- `Role` **belongs to** `Business` (nullable)
- `Role` **has many** `Users`

## 6. `categories`
**Purpose:** Group products for easier browsing and reporting.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | Scoped to business |
| name | string(255) | required | Category name |
| description | text | nullable | Optional description |
| sort_order | integer | default 0 | Display ordering |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** unique on `(business_id, name)`, index on `business_id`

**Relationships:**
- `Category` **belongs to** `Business`
- `Category` **has many** `Products`

## 7. `products`
**Purpose:** Inventory items available for sale.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | Scoped to business |
| category_id | bigInteger | FK → categories.id, nullable | Product group |
| name | string(255) | required | Product name |
| description | text | nullable | |
| sku | string(100) | nullable | Stock keeping unit, unique per business |
| barcode | string(100) | nullable | Scanner barcode |
| unit_price | decimal(10,2) | required, >= 0 | Selling price (UGX) |
| cost_price | decimal(10,2) | nullable, >= 0 | Purchase cost (profit calc) |
| stock_quantity | integer | default 0, >= 0 | Live stock count |
| low_stock_threshold | integer | default 5 | Alert when stock <= this |
| tax_percentage | decimal(5,2) | default 0 | Tax rate applied |
| is_active | boolean | default true | Available for sale? |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |
| deleted_at | timestamp | nullable | soft deletes |

**Indexes:** unique on `(business_id, sku)` (if sku not null), index on `business_id`, index on `category_id`

**Relationships:**
- `Product` **belongs to** `Business`
- `Product` **belongs to** `Category` (nullable)
- `Product` **has many** `SaleItems`
- `Product` **has many** `StockMovements`

## 8. `customers`
**Purpose:** Optional buyer tracking with purchase history.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | Scoped to business |
| name | string(255) | required | Customer name |
| phone | string(50) | required | Primary contact |
| email | string(255) | nullable | |
| total_purchases | decimal(12,2) | default 0 | Lifetime spend (UGX) |
| last_purchase_at | timestamp | nullable | Last purchase date |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** unique on `(business_id, phone)`, index on `business_id`

**Relationships:**
- `Customer` **belongs to** `Business`
- `Customer` **has many** `Sales`

## 9. `sales`
**Purpose:** Transaction header — one sale = one receipt.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | |
| user_id | bigInteger | FK → users.id, required | Who processed the sale |
| customer_id | bigInteger | FK → customers.id, nullable | Walk-in if null |
| shift_id | bigInteger | FK → shifts.id, nullable | Which shift this belongs to |
| receipt_number | string(50) | required | Auto-generated, unique per business |
| subtotal | decimal(12,2) | required | Sum of line item subtotals |
| tax_total | decimal(12,2) | default 0 | Total tax collected |
| discount_amount | decimal(12,2) | default 0 | Total discount applied |
| total_amount | decimal(12,2) | required | Final amount (subtotal + tax - discount) |
| payment_method | enum('cash','mobile_money','card','other') | required | UG market: Mobile Money is key |
| payment_status | enum('paid','partially_refunded','refunded') | default 'paid' | Refund tracking |
| notes | text | nullable | Staff notes |
| sale_date | datetime | required | When sale occurred (for reporting) |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |
| deleted_at | timestamp | nullable | soft deletes |

**Indexes:** unique on `(business_id, receipt_number)`, index on `business_id`, index on `user_id`, index on `customer_id`, index on `shift_id`, index on `sale_date`

**Relationships:**
- `Sale` **belongs to** `Business`
- `Sale` **belongs to** `User` (staff who processed it)
- `Sale` **belongs to** `Customer` (nullable)
- `Sale` **belongs to** `Shift` (nullable)
- `Sale` **has many** `SaleItems`

## 10. `sale_items`
**Purpose:** Individual line items within a sale. Also tracks refunds at the line-item level.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| sale_id | bigInteger | FK → sales.id, required | Parent sale |
| product_id | bigInteger | FK → products.id, nullable | Nullable on product delete |
| product_name | string(255) | required | Snapshot of product name at sale time |
| product_price | decimal(10,2) | required | Snapshot of price at sale time |
| quantity | integer | required, > 0 | Quantity sold |
| unit_price | decimal(10,2) | required | Price after any line discount |
| subtotal | decimal(12,2) | required | quantity × unit_price |
| tax_amount | decimal(12,2) | default 0 | Tax for this line |
| discount_amount | decimal(12,2) | default 0 | Line-level discount |
| refunded_quantity | integer | default 0 | How many of this line refunded |
| refunded_amount | decimal(12,2) | default 0 | Total refunded for this line |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** index on `sale_id`, index on `product_id`

**Relationships:**
- `SaleItem` **belongs to** `Sale`
- `SaleItem` **belongs to** `Product` (nullable)
- `SaleItem` **has many** `StockMovements` (linked via sale_item_id)

## 11. `stock_movements`
**Purpose:** Inventory ledger — every stock change is recorded with a before/after snapshot.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | |
| product_id | bigInteger | FK → products.id, required | Which product changed |
| sale_item_id | bigInteger | FK → sale_items.id, nullable | Linked if from a sale/refund |
| type | enum('purchase','sale','adjustment','return','initial') | required | What caused the change |
| quantity_change | integer | required | + for additions, - for reductions |
| stock_before | integer | required | Snapshot before change |
| stock_after | integer | required | Snapshot after change |
| reference | string(255) | nullable | e.g. PO-001, receipt # |
| notes | text | nullable | Reason for adjustment |
| created_by | bigInteger | FK → users.id, nullable | Who performed the action |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** index on `business_id`, index on `product_id`, index on `sale_item_id`, index on `created_by`, index on `type`

**Relationships:**
- `StockMovement` **belongs to** `Business`
- `StockMovement` **belongs to** `Product`
- `StockMovement` **belongs to** `SaleItem` (nullable)
- `StockMovement` **belongs to** `User` (as created_by)

## 12. `shifts`
**Purpose:** Track user work sessions — clock in/out and all sales made during the shift.

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | |
| user_id | bigInteger | FK → users.id, required | Which staff |
| clock_in | datetime | required | When shift started |
| clock_out | datetime | nullable | When shift ended |
| total_sales | decimal(12,2) | default 0 | Cached sum of sales in this shift |
| total_cash | decimal(12,2) | default 0 | Cached cash payments in shift |
| total_mobile_money | decimal(12,2) | default 0 | Cached mobile money in shift |
| total_card | decimal(12,2) | default 0 | Cached card payments in shift |
| status | enum('active','completed') | default 'active' | Currently working? |
| notes | text | nullable | Shift notes |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** index on `business_id`, index on `user_id`, index on `(business_id, user_id, status)` for "find active shift" query, index on `clock_in`

**Relationships:**
- `Shift` **belongs to** `Business`
- `Shift` **belongs to** `User`
- `Shift` **has many** `Sales`

## 13. `expense_categories`
**Purpose:** Categorise business expenses (e.g. Rent, Utilities, Salaries, Supplies).

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | Scoped to business |
| name | string(255) | required | e.g. "Rent", "Electricity" |
| description | text | nullable | |
| sort_order | integer | default 0 | Display ordering |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |

**Indexes:** unique on `(business_id, name)`, index on `business_id`

**Relationships:**
- `ExpenseCategory` **belongs to** `Business`
- `ExpenseCategory` **has many** `Expenses`

## 14. `expenses`
**Purpose:** Record business expenses for net sales calculation (Revenue − Expenses = Net).

| Column | Type | Constraints | Notes |
|--------|------|-------------|-------|
| id | bigIncrements | PK | |
| business_id | bigInteger | FK → businesses.id, required | |
| expense_category_id | bigInteger | FK → expense_categories.id, nullable | |
| recorded_by | bigInteger | FK → users.id, nullable | Who recorded the expense |
| amount | decimal(12,2) | required, > 0 | Expense amount (UGX) |
| description | text | required | What the expense was for |
| reference | string(255) | nullable | e.g. invoice #, receipt # |
| expense_date | datetime | required | When expense occurred |
| created_at | timestamp | nullable | |
| updated_at | timestamp | nullable | |
| deleted_at | timestamp | nullable | soft deletes |

**Indexes:** index on `business_id`, index on `expense_category_id`, index on `recorded_by`, index on `expense_date`

**Relationships:**
- `Expense` **belongs to** `Business`
- `Expense` **belongs to** `ExpenseCategory` (nullable)
- `Expense` **belongs to** `User` (as recorded_by)

---
## 🔗 Entity Relationship Summary

```
plans
  └── subscriptions        (1:N)

subscriptions
  ├── plan                 (N:1)
  └── business             (1:1)

businesses
  ├── owner                (N:1 — the owner user)
  ├── subscription         (1:1)
  ├── users                (1:N — staff)
  ├── roles                (1:N)
  ├── categories           (1:N)
  ├── products             (1:N)
  ├── customers            (1:N)
  ├── sales                (1:N)
  ├── stock_movements      (1:N)
  ├── shifts               (1:N)
  ├── expense_categories   (1:N)
  └── expenses             (1:N)

users
  ├── business             (N:1 — the business they work for)
  ├── owned_business       (1:1 — if they own one)
  ├── role                 (N:1)
  ├── created_by           (N:1 — self-referencing, who added them)
  ├── sales                (1:N — processed by)
  ├── shifts               (1:N)
  ├── stock_movements      (1:N — created_by)
  └── expenses             (1:N — recorded_by)

roles
  ├── business             (N:1, nullable)
  └── users                (1:N)

categories
  ├── business             (N:1)
  └── products             (1:N)

products
  ├── business             (N:1)
  ├── category             (N:1, nullable)
  ├── sale_items           (1:N)
  └── stock_movements      (1:N)

customers
  ├── business             (N:1)
  └── sales                (1:N)

sales
  ├── business             (N:1)
  ├── user                 (N:1)
  ├── customer             (N:1, nullable)
  ├── shift                (N:1, nullable)
  └── sale_items           (1:N)

sale_items
  ├── sale                 (N:1)
  ├── product              (N:1, nullable)
  └── stock_movements      (1:N)

stock_movements
  ├── business             (N:1)
  ├── product              (N:1)
  ├── sale_item            (N:1, nullable)
  └── created_by           (N:1 — user)

shifts
  ├── business             (N:1)
  ├── user                 (N:1)
  └── sales                (1:N)

expense_categories
  ├── business             (N:1)
  └── expenses             (1:N)

expenses
  ├── business             (N:1)
  ├── expense_category     (N:1, nullable)
  └── recorded_by          (N:1 — user)
```

---
## 🧭 UI Module Organisation

The UI is built around **6 core modules**. Each module has specific features accessible based on user permissions *and* plan restrictions.

### 1. 📊 Dashboard
**Permission check:** `reports.view`  
**Plan gate:** All plans

| Feature | Description |
|---------|-------------|
| Daily Sales | Today's total sales amount |
| Revenue | Total revenue (period: today/week/month) |
| Net Sales | Revenue − Expenses (Premium only — hidden for Free) |
| Top Products | Best-selling products (by qty sold) |
| Low Stock Alert | Products below threshold |
| Quick Stats | Total sales count, payment method breakdown |

### 2. 🧾 Sales
**Permission check:** `sales.create` (to sell), `sales.view` (to view)  
**Plan gate:** Discounts & refunds restricted on Free tier

| Feature | Description |
|---------|-------------|
| New Sale | POS checkout screen (select products, cart, pay) |
| Cart | Active cart with line items, subtotal |
| Discounts | Apply discounts (Pro+) |
| Print Receipt | Reprint any previous receipt |
| Sales History | List of all sales with filters |
| Refunds | Process refunds (Pro+) |

### 3. 📦 Inventory
**Permission check:** `inventory.view`  
**Plan gate:** Product limit enforced on Free (50 max)

| Feature | Description |
|---------|-------------|
| Add Product | Create new product (respects plan limit) |
| Stock Tracking | View stock movements ledger per product |
| Low Stock Alerts | Products needing restock |
| Categories | Manage product categories |
| Product List | Browse/edit all products |

### 4. 👥 Customers
**Permission check:** `customers.view`  
**Plan gate:** Customer limit enforced on Free (50 max)

| Feature | Description |
|---------|-------------|
| Customer List | All customers with search/filter |
| Purchase History | All sales by selected customer |
| Add Customer | Register new customer |

### 5. 💸 Expenses
**Permission check:** `expenses.view`  
**Plan gate:** Pro+ only — hidden for Free users

| Feature | Description |
|---------|-------------|
| Record Expense | Log a new expense |
| Expense Categories | Manage expense categories |
| Expense List | View/search expense history |

### 6. ⚙️ Settings
**Permission check:** `settings.view`  
**Plan gate:** All plans

| Feature | Description |
|---------|-------------|
| Business Profile | Edit business info, currency, receipt footer, logo |
| Subscription | View current plan, upgrade/downgrade |
| Staff Management | Add/remove users, assign roles (`users.create`) |
| Roles & Permissions | Manage roles (`settings.edit`) |

---

### Navigation Structure (Sidebar)

```
┌──────────────────────┐
│  CUSTOSELL            │
│  [Business Name]      │
│  [Plan: Pro]          │
├──────────────────────┤
│  📊 Dashboard         │
│  🧾 Sales             │
│     ├ New Sale        │
│     ├ History         │
│     └ Refunds (Pro+)  │
│  📦 Inventory         │
│     ├ Products        │
│     ├ Categories      │
│     └ Stock Ledger    │
│  👥 Customers         │
│     └ List            │
│  💸 Expenses (Pro+)   │
│     ├ Record          │
│     └ List            │
│  ⚙️ Settings           │
│     ├ Business        │
│     ├ Subscription    │
│     ├ Staff           │
│     └ Roles           │
├──────────────────────┤
│  👤 Staff Name        │
│  🕐 Clocked In        │
└──────────────────────┘
```

**Notes:**
- Items marked `(Pro+)` are hidden for Free tier users
- Staff users only see modules they have role permissions for
- Business owner (Admin role) sees everything their plan allows

---
## 📋 Feature Coverage Matrix

| Feature | Primary Table(s) | Permission Gate | Plan Gate |
|---------|-----------------|-----------------|-----------|
| 🏪 Business Registration | `businesses` | — | All |
| 💳 Subscription Management | `plans` + `subscriptions` | `settings.edit` | All |
| 👥 Add/Remove Staff | `users` + `roles` | `users.create/delete` | Pro+ (limit) |
| 🔐 Password Login | `users.password` | — | All |
| 🛡️ Role Management | `roles` (JSON permissions) | `settings.edit` | All |
| 🕐 Clock In/Out | `shifts` | `sales.create` | Pro+ |
| 📁 Categories | `categories` | `inventory.create/edit` | All (limit on Free) |
| 📦 Add Product | `products` | `inventory.create` | All (limit on Free) |
| 📦 Stock Tracking | `products.stock_quantity` | `inventory.view` | All |
| 🚨 Low Stock Alerts | `products` (stock <= threshold) | `inventory.view` | All |
| 🧾 POS Checkout (New Sale) | `sales` + `sale_items` | `sales.create` | All |
| 🛒 Cart | `sale_items` (in-memory then persisted) | `sales.create` | All |
| 🏷️ Discounts | `sales.discount_amount` + `sale_items.discount_amount` | `sales.discount` | Pro+ |
| 🧑 Customer List | `customers` | `customers.view` | All (limit on Free) |
| 📜 Purchase History | `sales` + `sale_items` by `customer_id` | `customers.view` | All |
| 💰 Payment Methods | `sales.payment_method` | — | All |
| 🖨️ Receipt Printing | `sales` + `sale_items` + `businesses` | `sales.view` | All |
| ↩️ Refunds | `sale_items` (refunded fields) + `stock_movements` | `sales.refund` | Pro+ |
| 📊 Daily Sales | `sales` + `sale_items` | `reports.view` | All |
| 👷 Shift Sales | `shifts` + `sales.shift_id` | `reports.view` | Pro+ |
| 📈 Revenue | `sales.total_amount` | `reports.view` | All |
| 🏆 Top Products | `sale_items` (group by product_id) | `reports.view` | All |
| 📉 Net Sales (Revenue − Expenses) | `sales` + `expenses` | `reports.view` | Pro+ |
| 💸 Record Expense | `expenses` | `expenses.create` | Pro+ |
| 🗂️ Expense Categories | `expense_categories` | `expenses.create/edit` | Pro+ |
| 🧾 Inventory Audit | `stock_movements` | `inventory.view` | All |
| 📤 Export Data | CSV generation | — | Premium only |
| ⚙️ Business Settings | `businesses` | `settings.edit` | All |

---
## 📐 Index Summary

| Table | Type | Columns |
|-------|------|---------|
| plans | PK | id |
| plans | unique | slug |
| subscriptions | PK | id |
| subscriptions | unique | business_id |
| subscriptions | FK | plan_id |
| subscriptions | index | status |
| businesses | PK | id |
| businesses | unique | slug |
| businesses | FK | owner_id |
| users | PK | id |
| users | unique | email |
| users | FK | business_id |
| users | FK | role_id |
| users | FK | created_by |
| roles | PK | id |
| roles | unique | (business_id, slug) |
| categories | PK | id |
| categories | unique | (business_id, name) |
| categories | FK | business_id |
| products | PK | id |
| products | unique | (business_id, sku) |
| products | FK | business_id |
| products | FK | category_id |
| customers | PK | id |
| customers | unique | (business_id, phone) |
| customers | FK | business_id |
| sales | PK | id |
| sales | unique | (business_id, receipt_number) |
| sales | FK | business_id |
| sales | FK | user_id |
| sales | FK | customer_id |
| sales | FK | shift_id |
| sales | index | sale_date |
| sale_items | PK | id |
| sale_items | FK | sale_id |
| sale_items | FK | product_id |
| stock_movements | PK | id |
| stock_movements | FK | business_id |
| stock_movements | FK | product_id |
| stock_movements | FK | sale_item_id |
| stock_movements | FK | created_by |
| stock_movements | index | type |
| shifts | PK | id |
| shifts | FK | business_id |
| shifts | FK | user_id |
| shifts | index | (business_id, user_id, status) |
| shifts | index | clock_in |
| expense_categories | PK | id |
| expense_categories | unique | (business_id, name) |
| expense_categories | FK | business_id |
| expenses | PK | id |
| expenses | FK | business_id |
| expenses | FK | expense_category_id |
| expenses | FK | recorded_by |
| expenses | index | expense_date |

---
## 📡 Offline-First Architecture

**Architecture:** Electron + React (desktop) → Laravel API (cloud)

### Core principle
The desktop app **IS the source of truth**. The cloud is a backup/sync hub.
All features work offline. No internet required for daily POS operations.

### Local database (SQLite via better-sqlite3)
- Mirrors the cloud schema locally
- Stored per-business on the user's machine
- All reads/writes hit SQLite first — instant, no network wait

### Sync (automatic, no manual button)

```
App opens → check internet
  ├── Online → sync in background (push local changes, pull cloud changes)
  │            Every N minutes OR on key actions (sale completed, product added)
  │            User sees green "Connected" indicator
  └── Offline → everything works locally
                 Queue local changes with sync_token tracking
                 User sees yellow "Offline — will sync when connected"
                 On reconnect → auto-sync all queued changes
```

### How sync works

**Push** (local → cloud):
- Sales, sale_items, expenses, stock_movements — append-only, no conflicts
- Products, customers — only if created/edited locally
- Receipt numbers generated locally with business prefix + increment

**Pull** (cloud → local):
- Products, categories, customers, roles — only what's newer than local
- Uses `updated_at` timestamps for delta sync
- Staff users, shifts — managed per-device

### Conflict strategy
- **Sales** — append-only, no conflict possible
- **Products** — last-write-wins (cloud wins if synced from another device, local wins if edited offline)
- **Stock** — additive operations are safe (sales decrement, purchases increment). Manual adjustments flagged for review

### Data volume expectation
- 50–200 products per business
- 20–100 sales per day
- **Megabytes per business** — sync is lightweight

### Sync queue table (local SQLite only)
```sql
CREATE TABLE sync_queue (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  table_name TEXT NOT NULL,        -- e.g. 'sales'
  record_id INTEGER NOT NULL,      -- local ID
  action TEXT NOT NULL,             -- 'create', 'update', 'delete'
  payload JSON NOT NULL,            -- full record data
  synced INTEGER DEFAULT 0,        -- 0 = pending, 1 = synced
  created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);
```

### What the user sees
| State | Indicator | Behaviour |
|-------|-----------|-----------|
| Online | 🟢 Connected | Auto-sync in background |
| Offline | 🟡 Offline — syncing when connected | Full functionality, queue changes |
| Reconnecting | 🔄 Syncing... | Push queue, pull updates, then 🟢 |

---
## ✅ Net Sales Calculation

**Formula:** `Net Sales = Total Revenue − Total Expenses`

```sql
-- Daily net sales
SELECT
    COALESCE(SUM(s.total_amount), 0) AS total_revenue,
    COALESCE(SUM(e.amount), 0) AS total_expenses,
    COALESCE(SUM(s.total_amount), 0) - COALESCE(SUM(e.amount), 0) AS net_sales
FROM businesses b
LEFT JOIN sales s ON s.business_id = b.id AND DATE(s.sale_date) = CURDATE() AND s.deleted_at IS NULL
LEFT JOIN expenses e ON e.business_id = b.id AND DATE(e.expense_date) = CURDATE() AND e.deleted_at IS NULL
WHERE b.id = ?
```

---
## 🧪 Plan Enforcement Strategy (Code Level)

No scattered if/else across controllers. Centralised checks:

### Model Helper
```php
// Business model
public function plan(): Plan
{
    return $this->subscription->plan;
}

public function canFeature(string $feature): bool
{
    return $this->plan()->features[$feature] ?? false;
}

public function maxLimit(string $limit): ?int
{
    return $this->plan()->limits[$limit] ?? null;  // null = unlimited
}

public function hasReachedLimit(string $limit, int $currentCount): bool
{
    $max = $this->maxLimit($limit);
    return $max !== null && $currentCount >= $max;
}
```

### Middleware (Route Groups)
```php
// CheckFeature middleware
public function handle(Request $request, Closure $next, string $feature)
{
    if (!$request->user()->business->canFeature($feature)) {
        return response()->json([
            'message' => "Feature not available on your plan. Upgrade to access."
        ], 403);
    }
    return $next($request);
}
```

### Route Example
```php
Route::middleware(['auth:sanctum', 'check.feature:expenses'])->group(function () {
    Route::apiResource('expenses', ExpenseController::class);
});
```

### Frontend
- Hide/disable UI elements based on `business.plan.features` returned in auth response
- Show upgrade prompts instead of greyed-out features

---
## ✅ Ready for Review (v2.0)

**14 tables total**

### What's new in v2.1:
| Change | Details |
|--------|---------|
| ➕ `plans` table | 3 tiers: Free (UGX 0), Pro (UGX 30,000), Premium (UGX 100,000) |
| ➕ `subscriptions` table | Business ↔ Plan linking with status & dates |
| ➕ `businesses.owner_id` | Explicit FK → users.id for ownership |
| ➕ `businesses.trial_ends_at` | Quick trial tracking |
| 📡 Offline-first architecture | SQLite local DB, auto-sync, queue table |
| 🔄 Default currency | Changed from KES → **UGX** |
| 🔄 `mpesa` → `mobile_money` | Payment method more inclusive for UG market |
| 🧪 Plan enforcement | Middleware + model helpers — no scattered if/else |
| 🔒 Feature + permission gating | Every feature has a plan gate AND a role permission |
| 🧭 UI sidebar | Items marked `(Pro+)` hidden for Free users |

### 3 plans:
| | 🆓 Free | ⭐ Pro | 🏢 Premium |
|---|---|---|---|
| **Price** | UGX 0 | UGX 30,000/mo | UGX 100,000/mo |
| **Staff** | 1 | 5 | Unlimited |
| **Products** | 50 | 1,000 | Unlimited |
| **Expenses** | ❌ | ✅ | ✅ |
| **Shifts** | ❌ | ✅ | ✅ |
| **Discounts** | ❌ | ✅ | ✅ |
| **Refunds** | ❌ | ✅ | ✅ |
| **Export** | ❌ | ❌ | CSV |